<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_7_QM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Quantum chemistry with ORCA

First save a copy of this notebook in your Google Drive (`File -> Save a copy in Drive`).

This is a group report. As in the training notebook (`t07_QM`), the two groups work on different topics:

*   **GROUP 1: the forces that hold biomolecules together.** How strong are a hydrogen bond between two DNA bases, the stacking of two bases on top of each other, or the contact between two aromatic side chains in a protein? Where does this energy come from, and how well do simple models describe it?
*   **GROUP 2: the colour of biomolecules.** Why is the green fluorescent protein green, why is rhodopsin (the protein of vision) purple, and why are flavoproteins yellow? We compute the light absorbed by their chromophores, and how it depends on their chemistry and environment.

The notebook has two parts for each group:

1.  **Running the calculations**: each member of the group studies one or two molecules (see the tables below), running the cells of Part 1 with his or her own choice.
2.  **The report**: the analysis of the results of the whole group, with your text. The code cells collect the results of all the members and make the figures; write your answers in the text cells, in your own words.

**You will present your work to the class** (about 10 minutes per group, followed by questions; the other group has not studied your topic). For this reason the report asks you, at each step, to make figures that explain what you did. The figures made with matplotlib are saved in the folder `figures` of the group; for the 3D views (molecules, orbitals, proteins) rotate the molecule to a clear orientation and take a screenshot, or open the files in VMD or ChimeraX.

The cells that start with `%%group 1` or `%%group 2` run only for the group you choose below, so you can also use *Runtime -> Run all*.

**Group:** 1 / 2

**Members:**

In [ ]:
from datetime import date

today = date.today()
print(today)

## The assignment

### GROUP 1: the forces that hold biomolecules together

The double helix of DNA, the folded structure of a protein, the binding of a drug to its target: all are held together by **noncovalent interactions**, much weaker than covalent bonds (≈ 100 kcal/mol) but numerous. Each is worth a few kcal/mol, only a few times the thermal energy $k_BT$ = 0.6 kcal/mol: strong enough to give stable structures, weak enough to be broken and formed again during function. In a force field they are described by two simple terms, point charges (Coulomb) and Lennard-Jones; with quantum mechanics we can compute them from the electrons and understand where they come from:

| ingredient | what it is | in a force field | in Hartree–Fock |
|:--|:--|:--|:--|
| **electrostatics** | attraction between the positive and negative parts of the two molecules (e.g. the partially positive H of an N–H and the lone pair of an O) | point charges | yes |
| **polarisation** | each molecule deforms the electron cloud of the other | missing (or averaged in the charges) | yes |
| **dispersion** (London forces) | the fluctuating electron clouds of the two molecules attract each other; always attractive, important for large flat molecules | the attractive $-1/r^6$ part of Lennard-Jones | **no** |
| **Pauli repulsion** | electron clouds cannot overlap | the repulsive $1/r^{12}$ part of Lennard-Jones | yes |

The two main types of interaction you will compare are the **hydrogen bond** (directional, mostly electrostatic: base pairing, secondary structure) and **stacking** (flat aromatic rings on top of each other, mostly dispersion: neighbouring bases in DNA, aromatic side chains in the core of proteins).

We use the complexes of the **S22 database** (Jurečka et al., *Phys. Chem. Chem. Phys.* 8, 1985 (2006)): small models of biological interactions with very accurate reference interaction energies (CCSD(T) extrapolated to an infinite basis set, Takatani et al., *J. Chem. Phys.* 132, 144104 (2010)), also at five distances between the two molecules (S22x5, Gráfová et al., *J. Chem. Theory Comput.* 6, 2365 (2010)). The reference values are our "experiment". **Each member takes one pair of complexes**, the same type of molecules in two arrangements:

| member | complex 1 | complex 2 | where you find them in biology |
|:--:|:--|:--|:--|
| 1 | adenine–thymine, Watson–Crick (S22 n. 7) | adenine–thymine, stacked (15) | base pairing vs base stacking in DNA |
| 2 | uracil dimer, H-bonded (5) | uracil dimer, stacked (13) | the same in RNA |
| 3 | indole–benzene, stacked (14) | indole–benzene, T-shaped (21) | contacts between Trp and Phe/Tyr in the core of proteins |
| 4 | formamide dimer (4) | formic acid dimer (3) | the N–H···O=C bonds of the protein backbone; carboxylic groups (Asp, Glu) |
| 5 | phenol dimer (22) | benzene–water (17) | Tyr–Tyr contacts; O–H groups pointing to an aromatic ring |

With fewer than five members, choose the pairs you find most interesting (members 1 and 3 are the most instructive); with more, two members can share a pair.

**What each member computes** (Part 1): the interaction energy of the two complexes with methods of increasing quality (HF, B3LYP, B3LYP with the D4 dispersion correction, r$^2$SCAN-3c, all corrected for the BSSE as in the training); the effect of the basis set; the energy as a function of the distance between the molecules; the atomic charges and the electrostatic potential of the two molecules. **Time**: 1–2 hours for the largest complexes (adenine–thymine, 30 atoms), 10–30 minutes for the smaller ones. Start early.

### GROUP 2: the colour of biomolecules

Proteins and nucleic acids are colourless: their electrons need ultraviolet light to be excited (training, Part 6). Nature makes colour with **chromophores**, molecules with long chains of alternating single and double bonds (*conjugated* systems), whose excitation energy falls in the visible. Often the chromophore is bound inside a protein, which tunes its colour: the same retinal molecule absorbs blue, green or red light in the different opsins of our cones.

The quantity we compute is the **excitation energy** (or the wavelength $\lambda$ = 1239.84 / E(eV)) of the excited states and their **oscillator strength** $f$ (how intense the absorption is), with TD-DFT as in the training. From them we get the absorption spectrum and the colour. We will ask:

*   which part of the molecule absorbs the light: which orbitals are involved (HOMO, LUMO), and where are they?
*   how the colour changes with the chemistry: protonation, length of the conjugated chain, oxidation;
*   how it changes with the environment: vacuum vs water;
*   how reliable the calculation is: the result depends on the approximate functional, and we compare with experiment.

**Each member takes one system:**

| member | molecules (files in `Data/qm/chromophores`) | biological question | experimental $\lambda_{max}$ (approximate, check and cite) |
|:--:|:--|:--|:--|
| 1 | GFP chromophore (HBDI): neutral, anion (phenolate), cation | how does the protonation state change the colour of GFP? | water: neutral ≈ 370 nm, anion ≈ 425 nm, cation ≈ 395 nm; anion in vacuo 479 nm; in GFP 395 nm (neutral) and 475 nm (anion) |
| 2 | retinal protonated Schiff base: all-*trans* and 11-*cis* | why is retinal coloured, and how does the protein tune it? | in vacuo ≈ 610 nm; in methanol ≈ 440 nm; rhodopsin (11-*cis*) 498 nm; bacteriorhodopsin (all-*trans*) 568 nm |
| 3 | conjugated chains (protonated Schiff bases with 2 to 6 double bonds, `psb_2` … `psb_6`) | how does the length of the conjugated chain decide the colour? | retinal has 6 conjugated double bonds |
| 4 | lumiflavin, oxidised and reduced | why do flavoproteins lose their yellow colour when they are reduced? | oxidised, water: ≈ 445 and ≈ 370 nm; reduced: no band in the visible |
| 5 | models of the aromatic side chains: toluene (Phe), *p*-cresol (Tyr), *p*-cresolate (deprotonated Tyr), 3-methylindole (Trp) | why do proteins absorb light at 280 nm? | Phe ≈ 257 nm, Tyr ≈ 275 nm, tyrosinate ≈ 293 nm, Trp ≈ 280 nm (weak bands) |

With fewer than five members, choose the systems you find most interesting (members 1, 2 and 4 make a good set); with more, two members can share a system and test more functionals.

**What each member computes** (Part 1): the structure of the molecules (with the fast semiempirical method GFN2-xTB, as in the dynamics of the training); the excited states with TD-DFT (CAM-B3LYP) in vacuum and in water (as a continuum, CPCM), with the HOMO and LUMO; the same with three other functionals, to see how much the result depends on this choice. **Time**: a few minutes per molecule for the small systems, about 1.5 hours for retinal (54 atoms). Start early.

### Organising the work

To share the results, one member creates a folder `Task7_group1` (or `Task7_group2`) in Google Drive and shares it with the rest of the group, with editing rights. Every member adds a shortcut to it in his or her own `MyDrive` (right click on the folder -> `Organise` -> `Add shortcut`). The calculations run on the local disk of Colab and their results are copied automatically to this folder; each molecule has its own sub-folder, so different members never write the same files. Each member also needs his or her own copy of ORCA in `MyDrive/Structural_Bioinformatics/ORCA`, as in the training.

If Colab disconnects, run the setup cells again and then the cell that was interrupted: the calculations already finished are not repeated.

# Setup (run these cells at the start of every session)

In [ ]:
GROUP = 1      # 1 (noncovalent interactions) or 2 (colour of chromophores)

In [ ]:
import os, glob, shutil, subprocess

ORCA_DIR = os.environ.get('ORCA_DIR', '')      # on your own computer: the folder containing the program orca
DRIVE_FOLDER = '/content/drive/MyDrive/Structural_Bioinformatics'

if not (ORCA_DIR and os.path.isfile(os.path.join(ORCA_DIR, 'orca'))):
    from google.colab import drive
    drive.mount('/content/drive')
    archives = sorted(glob.glob(os.path.join(DRIVE_FOLDER, 'ORCA', 'orca_6*linux_x86-64*.tar.*')))
    if not archives:
        raise FileNotFoundError('No ORCA archive found in ' + os.path.join(DRIVE_FOLDER, 'ORCA') +
                                ': follow the instructions above to download it.')
    archive = archives[-1]                        # if there is more than one, take the most recent version
    ORCA_DIR = os.path.join('/content', os.path.basename(archive).split('.tar')[0])
    if not os.path.isfile(os.path.join(ORCA_DIR, 'orca')):
        print('unpacking', os.path.basename(archive), '...')
        subprocess.run(['tar', '-xf', archive, '-C', '/content'], check=True)
    # some archives contain an additional folder level
    found = glob.glob(os.path.join('/content', 'orca_6*', '**', 'orca'), recursive=True)
    ORCA_DIR = os.path.dirname(sorted(found)[-1])

# ORCA must be called with its full path; the shared version also needs its libraries
os.environ['PATH'] = ORCA_DIR + ':' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = ORCA_DIR + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OMP_NUM_THREADS'] = '1'
ORCA = os.path.join(ORCA_DIR, 'orca')
version = subprocess.run([ORCA, '--version'], capture_output=True, text=True).stdout.strip()
print('ORCA found in', ORCA_DIR)
print(version if version else 'ORCA is installed')

In [ ]:
%pip install -q py3Dmol

The results are computed in a folder of the virtual machine and copied to the group folder `Task7_groupN` on Google Drive. The function `sync_from_drive()` brings to the virtual machine the results of the other members.

In [ ]:
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()
if os.path.isdir('/content/drive/MyDrive'):          # on Colab: compute on the local disk, keep the results in the group folder
    local_path = f'/content/T07_group{GROUP}'
    DRIVE_PATH = f'/content/drive/MyDrive/Task7_group{GROUP}'
    os.makedirs(DRIVE_PATH, exist_ok=True)
else:                                                # on your computer: a folder next to the notebook
    local_path = os.path.join(BASE_DIR, f'T07_group{GROUP}')
    DRIVE_PATH = None
os.makedirs(local_path, exist_ok=True)

def sync_from_drive():
    # copy to the virtual machine the results of the whole group (also those computed by the other members)
    if DRIVE_PATH:
        shutil.copytree(DRIVE_PATH, local_path, dirs_exist_ok=True)

sync_from_drive()
%cd $local_path
print('results of the group are in', DRIVE_PATH if DRIVE_PATH else local_path)

# the input structures are in the Data folder of the course repository
if os.path.isdir(os.path.join(BASE_DIR, '..', 'Data', 'qm', 's22')):
    REPO_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
else:
    REPO_DIR = '/content/sb' if os.path.isdir('/content') else os.path.join(BASE_DIR, 'sb')
    if os.path.isdir(REPO_DIR):
        !cd "$REPO_DIR" && git pull -q
    else:
        !git clone -q --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git "$REPO_DIR"
QM_DIR = os.path.join(REPO_DIR, 'Data', 'qm')
print('input files from', QM_DIR)

The helper functions of the training, and a few new ones (read them: they are short).

In [ ]:
import re, time
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol

HARTREE2EV = 27.211386        # 1 Eh in eV
HARTREE2KCAL = 627.5095       # 1 Eh in kcal/mol
BOHR2ANG = 0.529177           # 1 bohr in Å

def save_to_drive(workdir):
    # copy the useful files of a calculation (not the large temporary ones) to the same folder on Google Drive
    if not DRIVE_PATH:
        return
    target = os.path.join(DRIVE_PATH, os.path.relpath(os.path.abspath(workdir), local_path))
    os.makedirs(target, exist_ok=True)
    for f in os.listdir(workdir):
        if f.endswith(('.inp', '.out', '.xyz', '.cube', '.csv', '.hess', '.json', '.png')):
            shutil.copy2(os.path.join(workdir, f), target)

def run_orca(name, inp, folder='.', rerun=False):
    # write folder/name/name.inp, run ORCA in folder/name and return the path of the output file
    workdir = os.path.join(folder, name)
    os.makedirs(workdir, exist_ok=True)
    out = os.path.join(workdir, name + '.out')
    inpfile = os.path.join(workdir, name + '.inp')
    inp = inp.strip() + '\n'
    if (not rerun and os.path.isfile(out) and 'ORCA TERMINATED NORMALLY' in open(out).read()
            and os.path.isfile(inpfile) and open(inpfile).read() == inp):
        print(f'{name}: already done, reading {out}')
        return out
    for f in os.listdir(workdir):            # a new run starts from an empty folder
        if os.path.isfile(os.path.join(workdir, f)):
            os.remove(os.path.join(workdir, f))
    with open(inpfile, 'w') as f:
        f.write(inp)
    t0 = time.time()
    with open(out, 'w') as f:
        subprocess.run([ORCA, name + '.inp'], cwd=workdir, stdout=f, stderr=subprocess.STDOUT)
    text = open(out).read()
    if 'ORCA TERMINATED NORMALLY' not in text:
        print(text[-3000:])
        raise RuntimeError(f'ORCA did not terminate normally: look at the end of {out}')
    print(f'{name}: done in {time.time() - t0:.1f} s')
    save_to_drive(workdir)
    return out

def xyz_block(xyzfile, ghost=()):
    # the coordinate lines of an xyz file; the atoms whose index is in ghost become ghost atoms (see Part 4)
    lines = open(xyzfile).read().splitlines()
    natoms = int(lines[0])
    atoms = []
    for i, l in enumerate(lines[2:2 + natoms]):
        el, x, y, z = l.split()[:4]
        atoms.append(f'{el}{":" if i in ghost else " "} {x:>12} {y:>12} {z:>12}')
    return '\n'.join(atoms)

def final_energy(out):
    # the last 'FINAL SINGLE POINT ENERGY' of an output (in Eh)
    return float(re.findall(r'FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)', open(out).read())[-1])

def wall_time(out):
    # total run time of a calculation, in seconds
    d, h, m, s, ms = re.findall(r'TOTAL RUN TIME:\s+(\d+) days (\d+) hours (\d+) minutes (\d+) seconds (\d+) msec', open(out).read())[-1]
    return 86400*int(d) + 3600*int(h) + 60*int(m) + int(s) + int(ms)/1000

def s2(out):
    # expectation value of S^2 (only for UHF/UKS calculations)
    return float(re.findall(r'Expectation value of <S\*\*2>\s*:\s*(-?\d+\.\d+)', open(out).read())[-1])

def dipole(out):
    # magnitude of the dipole moment, in Debye
    return float(re.findall(r'Magnitude \(Debye\)\s*:\s*(\d+\.\d+)', open(out).read())[-1])

def orbital_energies(out):
    # the last table of orbital energies: list of (occupation, energy in eV); for UHF the spin-up orbitals
    text = open(out).read().split('ORBITAL ENERGIES')[-1]
    rows = []
    for l in text.splitlines()[4:]:
        f = l.split()
        if len(f) != 4:
            if rows: break
            continue
        rows.append((float(f[1]), float(f[3])))
    return rows

def charges(out, kind='MULLIKEN'):
    # atomic charges: kind is MULLIKEN, LOEWDIN, HIRSHFELD (needs the keyword Hirshfeld) or CHELPG (keyword CHELPG)
    text = open(out).read()
    if kind == 'MULLIKEN':
        block = text.split('MULLIKEN ATOMIC CHARGES')[-1].split('Sum of atomic charges')[0]
    elif kind == 'LOEWDIN':
        block = text.split('LOEWDIN ATOMIC CHARGES')[-1].split('LOEWDIN REDUCED')[0]
    elif kind == 'HIRSHFELD':
        block = text.split('HIRSHFELD ANALYSIS')[-1].split('TOTAL')[0]
    elif kind == 'CHELPG':
        block = text.split('CHELPG Charges')[-1].split('Total charge')[0]
    return [(el, float(q)) for el, q in re.findall(r'^\s*\d+\s+([A-Z][a-z]?)\s*:?\s+(-?\d+\.\d+)', block, re.M)]

def read_xyz(xyzfile):
    lines = open(xyzfile).read().splitlines()
    n = int(lines[0])
    el = [l.split()[0] for l in lines[2:2+n]]
    xyz = np.array([[float(v) for v in l.split()[1:4]] for l in lines[2:2+n]])
    return el, xyz

def read_cube(cubefile):
    # Gaussian cube file -> (origin, axes, values); origin and axes in bohr
    with open(cubefile) as f:
        f.readline(); f.readline()
        n, *origin = f.readline().split()
        natoms = abs(int(n))
        axes, shape = [], []
        for _ in range(3):
            npt, *v = f.readline().split()
            shape.append(int(npt)); axes.append([float(x) for x in v])
        for _ in range(natoms):
            f.readline()
        data = np.array(f.read().split(), dtype=float)
    return np.array(origin, dtype=float), np.array(axes), data[-np.prod(shape):].reshape(shape)

def show_molecule(xyzfile, width=400, height=300):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(open(xyzfile).read(), 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.zoomTo()
    return view.show()

def cube_for_display(cubefile, maxpoints=40):
    # the text of a cube file with at most maxpoints per side and 4 digits: much smaller in the notebook
    with open(cubefile) as f:
        lines = f.readlines()
    natoms = abs(int(lines[2].split()[0]))
    origin, axes, data = read_cube(cubefile)
    step = [max(1, int(np.ceil(n / maxpoints))) for n in data.shape]
    data = data[::step[0], ::step[1], ::step[2]]
    head = lines[:2] + [lines[2]]
    for i in range(3):
        v = axes[i] * step[i]
        head.append(f'{data.shape[i]:5d} {v[0]:12.6f} {v[1]:12.6f} {v[2]:12.6f}\n')
    head += lines[6:6 + natoms]
    values = data.ravel()
    body = '\n'.join(' '.join(f'{x:.4g}' for x in values[k:k + 6]) for k in range(0, len(values), 6))
    return ''.join(head) + body + '\n'

def cube_atoms_xyz(cubefile):
    # the atoms stored in a cube file (in bohr) as an xyz text (in Å)
    elements = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 16: 'S'}
    with open(cubefile) as f:
        lines = f.readlines()
    natoms = abs(int(lines[2].split()[0]))
    atoms = []
    for l in lines[6:6 + natoms]:
        z, _, x, y, w = l.split()[:5]
        atoms.append(f'{elements.get(int(float(z)), "X")} {float(x) * BOHR2ANG:.4f} {float(y) * BOHR2ANG:.4f} {float(w) * BOHR2ANG:.4f}')
    return f'{natoms}\n\n' + '\n'.join(atoms) + '\n'

def show_cube(cubefile, isovalue=0.05, signed=False, width=400, height=300):
    # isosurface of a cube file; signed=True for orbitals (positive lobes red, negative lobes blue)
    view = py3Dmol.view(width=width, height=height)
    view.addModel(cube_atoms_xyz(cubefile), 'xyz')
    view.setStyle({'stick': {'radius': 0.12}, 'sphere': {'scale': 0.2}})
    data = cube_for_display(cubefile)
    view.addVolumetricData(data, 'cube', {'isoval': isovalue, 'color': 'red', 'opacity': 0.8})
    if signed:
        view.addVolumetricData(data, 'cube', {'isoval': -isovalue, 'color': 'blue', 'opacity': 0.8})
    view.zoomTo()
    return view.show()

def plots_block(cubes, npoints=50):
    # an ORCA %plots block that writes cube files; cubes is a list of strings like
    # 'ElDens("x.eldens.cube")' or 'MO("x.mo4.cube",4,0)' (orbitals are numbered from 0)
    return '%plots\n  dim1 {0}\n  dim2 {0}\n  dim3 {0}\n  Format Gaussian_Cube\n'.format(npoints) + \
           ''.join(f'  {c};\n' for c in cubes) + 'end\n'

In [ ]:
def interaction_energy(xyzfile, nA, method, folder, counterpoise=True, q=(0, 0, 0)):
    # interaction energy (kcal/mol) of a dimer whose first nA atoms are molecule A and the others molecule B
    # q = (charge of AB, charge of A, charge of B); all species are closed-shell singlets
    nat = int(open(xyzfile).readline())
    tag = os.path.basename(xyzfile).replace('.xyz', '') + '_' + re.sub(r'[^A-Za-z0-9_+-]', '', method.replace(' ', '_'))   # no spaces, / or () in file names
    A, B = range(nA), range(nA, nat)
    jobs = {'AB': (xyz_block(xyzfile), q[0])}
    if counterpoise:                                     # monomers in the basis of the dimer: the partner is made of ghost atoms
        jobs['A'] = (xyz_block(xyzfile, ghost=B), q[1])
        jobs['B'] = (xyz_block(xyzfile, ghost=A), q[2])
    else:                                                # monomers in their own basis: the partner is removed
        lines = xyz_block(xyzfile).splitlines()
        jobs['A'] = ('\n'.join(lines[:nA]), q[1])
        jobs['B'] = ('\n'.join(lines[nA:]), q[2])
    E = {}
    for k, (coords, charge) in jobs.items():
        name = f'{tag}_{k}' + ('_cp' if counterpoise and k != 'AB' else '')
        out = run_orca(name, f'! {method} TightSCF\n* xyz {charge} 1\n{coords}\n*', folder=folder)
        E[k] = final_energy(out)
    return (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL

In [ ]:
def excitations(out):
    # excited states from the absorption spectrum table: list of (energy eV, wavelength nm, oscillator strength)
    text = open(out).read().split('ABSORPTION SPECTRUM VIA TRANSITION ELECTRIC DIPOLE MOMENTS')[1].split('\n\n')[0]
    rows = []
    for l in text.splitlines():
        if '->' in l:
            f = l.split('->')[1].split()
            rows.append((float(f[1]), float(f[3]), float(f[4])))
    return np.array(rows)

def transitions(out):
    # the orbital contributions to each excited state, as printed by ORCA (e.g. "7a -> 8a : 0.98")
    text = open(out).read().split('EXCITED STATES')[1].split('ABSORPTION SPECTRUM')[0]
    for block in re.split(r'\n(?=STATE\s+\d+:)', text)[1:]:
        head = block.splitlines()[0]
        contr = re.findall(r'(\d+a\s+->\s+\d+a)\s+:\s+(\d\.\d+)', block)
        print(head.split('<S**2>')[0].strip(), '  ', ',  '.join(f'{t} ({c})' for t, c in contr))

def spectrum(states, fwhm=0.3, emin=1.5, emax=8.0):
    # absorption spectrum: a Gaussian of width fwhm (eV) for each state, with area proportional to f
    E = np.linspace(emin, emax, 1000)
    sigma = fwhm / 2.3548
    eps = sum(f * np.exp(-(E - e)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi)) for e, lam, f in states)
    return E, eps

def colour_of(states, fwhm=0.3, max_absorbance=1.0):
    # colour of the light transmitted by a solution of the molecule (white light in, absorbed bands removed),
    # with the concentration chosen so that the strongest band has absorbance max_absorbance
    lam = np.arange(380.0, 781.0, 1.0)                        # visible wavelengths (nm)
    def g(x, mu, s1, s2):                                    # CIE 1931 colour matching functions (Wyman et al. 2013)
        return np.exp(-0.5 * ((x - mu) / np.where(x < mu, s1, s2))**2)
    xbar = 1.056 * g(lam, 599.8, 37.9, 31.0) + 0.362 * g(lam, 442.0, 16.0, 26.7) - 0.065 * g(lam, 501.1, 20.4, 26.2)
    ybar = 0.821 * g(lam, 568.8, 46.9, 40.5) + 0.286 * g(lam, 530.9, 16.3, 31.1)
    zbar = 1.217 * g(lam, 437.0, 11.8, 36.0) + 0.681 * g(lam, 459.0, 26.0, 13.8)
    sigma = fwhm / 2.3548
    E = 1239.84 / lam
    band = lambda e: np.exp(-(E - e)**2 / (2 * sigma**2))
    E_all = np.linspace(1.0, 12.0, 4000)
    peak = max(sum(f * np.exp(-(E_all - e)**2 / (2 * sigma**2)) for e, _, f in states).max(), 1e-12)
    absorbance = max_absorbance * sum(f * band(e) for e, _, f in states) / peak
    T = 10**(-absorbance)
    XYZ = np.array([np.sum(T * c) for c in (xbar, ybar, zbar)]) / np.sum(ybar)
    M = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])
    white = M @ (np.array([np.sum(c) for c in (xbar, ybar, zbar)]) / np.sum(ybar))
    rgb = np.clip(M @ XYZ / white, 0, 1)                     # white light -> (1, 1, 1)
    rgb = np.where(rgb <= 0.0031308, 12.92 * rgb, 1.055 * rgb**(1 / 2.4) - 0.055)
    return np.clip(rgb, 0, 1)

In [ ]:
import json, glob, linecache, warnings
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from IPython.core.magic import register_cell_magic
warnings.filterwarnings('ignore')

@register_cell_magic
def group(line, cell):
    # a cell that starts with %%group N runs only if GROUP is N
    ipython = get_ipython()
    if int(line) != ipython.user_ns.get('GROUP'):
        print(f'This cell is for group {line}: skipped.')
        return
    code = ipython.transform_cell(cell)
    filename = f'<group {line} cell>'
    linecache.cache[filename] = (len(code), None, code.splitlines(True), filename)   # so that errors show the code
    exec(compile(code, filename, 'exec'), ipython.user_ns)

def save_figure(fig, name):
    # save a matplotlib figure for your presentation, in the folder figures of the group (also on Google Drive)
    os.makedirs(os.path.join(local_path, 'figures'), exist_ok=True)
    fig.savefig(os.path.join(local_path, 'figures', name + '.png'), dpi=200, bbox_inches='tight')
    save_to_drive(os.path.join(local_path, 'figures'))
    print('saved figures/' + name + '.png')

def save_json(data, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w') as f:
        json.dump(data, f, indent=1)
    save_to_drive(os.path.dirname(filename))

def dispersion(out):
    # the D3/D4 dispersion correction (Eh) included in the final energy; 0 if there is none
    v = re.findall(r'Dispersion correction\s+(-?\d+\.\d+)', open(out).read())
    return float(v[-1]) if v else 0.0

def interaction_energy_split(xyzfile, nA, method, folder, q=(0, 0, 0)):
    # counterpoise-corrected interaction energy (kcal/mol), split in two parts: without the dispersion
    # correction, and the dispersion correction (so one B3LYP-D4 calculation gives both B3LYP and B3LYP-D4)
    nat = int(open(xyzfile).readline())
    tag = os.path.basename(xyzfile).replace('.xyz', '') + '_' + re.sub(r'[^A-Za-z0-9_+-]', '', method.replace(' ', '_'))
    A, B = range(nA), range(nA, nat)
    E, D = {}, {}
    for k, ghost, charge in [('AB', (), q[0]), ('A', B, q[1]), ('B', A, q[2])]:
        name = f'{tag}_{k}' + ('_cp' if k != 'AB' else '')
        out = run_orca(name, f'! {method} TightSCF\n* xyz {charge} 1\n{xyz_block(xyzfile, ghost=ghost)}\n*', folder=folder)
        E[k], D[k] = final_energy(out), dispersion(out)
    total = (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL
    disp = (D['AB'] - D['A'] - D['B']) * HARTREE2KCAL
    return total - disp, disp

def show_pdb_context(pdb, highlights, title='', labels=(), lines=(), local_file=None, width=700, height=420):
    # a structure from the PDB (downloaded by your browser from rcsb.org) or from a local file, with some residues
    # highlighted as sticks; highlights: list of 3Dmol selections, e.g. {'chain': 'A', 'resi': 6}
    if local_file:
        view = py3Dmol.view(width=width, height=height)
        view.addModel(open(local_file).read(), 'pdb')
    else:
        view = py3Dmol.view(query='pdb:' + pdb, width=width, height=height)
    view.setStyle({'cartoon': {'color': 'lightgrey', 'opacity': 0.8}})
    view.addStyle({'resn': ['DA', 'DT', 'DG', 'DC', 'A', 'U', 'G', 'C']}, {'stick': {'radius': 0.08, 'color': 'lightgrey'}})
    for k, sel in enumerate(highlights):
        view.addStyle(sel, {'stick': {'radius': 0.22, 'colorscheme': ['greenCarbon', 'orangeCarbon', 'cyanCarbon', 'magentaCarbon'][k % 4]}})
    for sel in labels:
        view.addResLabels(sel, {'fontSize': 11, 'showBackground': True, 'backgroundOpacity': 0.6})
    for p, q in lines:
        view.addCylinder({'start': dict(zip('xyz', map(float, p))), 'end': dict(zip('xyz', map(float, q))),
                          'radius': 0.1, 'dashed': True, 'color': 'darkorange'})
    view.zoomTo({'or': highlights} if len(highlights) > 1 else highlights[0] if highlights else {})
    print(title)
    return view.show()

# GROUP 1, Part 1: running the calculations

Write the S22 numbers of your two complexes (see the table above) and run the cells below. The first one shows the two complexes: in the xyz files the atoms of molecule A come first, then those of molecule B.

In [ ]:
%%group 1
my_complexes = [7, 15]          # the S22 numbers of your two complexes
BSSE_COMPLEX = my_complexes[0]  # the complex for which the large basis set (def2-TZVP) is also used

s22 = pd.read_csv(os.path.join(QM_DIR, 's22', 's22x5_reference.csv'))
SCALES = [0.9, 1.0, 1.2, 1.5, 2.0]

def s22_file(i, scale=1.0):
    row = s22[(s22.id == i) & (s22.scale == scale)].iloc[0]
    return os.path.join(QM_DIR, 's22', row.file), int(row.nA), float(row.Eint_ref_kcal)

for i in my_complexes:
    xyz, nA, ref = s22_file(i)
    print(f'S22 n. {i}: {s22[s22.id == i].system.iloc[0]}, reference interaction energy {ref:.2f} kcal/mol')
    show_molecule(xyz)

### The electrostatic potential: where are the positive and negative parts?

To see the electrostatics, we compute for each molecule (alone, at the geometry it has in the complex) the electron density and the **electrostatic potential** (ESP): the energy that a positive charge would feel at each point of space. The ESP is drawn with colours on the surface of the molecule (where the density is 0.002 electrons/bohr$^3$, close to the van der Waals surface): **red** where it is negative (electron-rich: lone pairs, $\pi$ clouds of aromatic rings), **blue** where it is positive (electron-poor: H atoms bound to N or O). The program `orca_vpot` computes it on the points of the density grid. The same calculation gives the CHELPG atomic charges, those fitted to this potential.

In [ ]:
%%group 1
def esp_cubes(name, coords, charge, folder, npoints=50):
    # density and electrostatic potential (cube files) of a molecule, and its output (with the CHELPG charges)
    workdir = os.path.join(folder, name)
    dens, esp = os.path.join(workdir, name + '.dens.cube'), os.path.join(workdir, name + '.esp.cube')
    inp = '! r2SCAN-3c CHELPG TightSCF KeepDens\n' + plots_block([f'ElDens("{name}.dens.cube")'], npoints) + \
          f'* xyz {charge} 1\n{coords}\n*'
    rerun = not os.path.isfile(esp) and not os.path.isfile(os.path.join(workdir, name + '.gbw'))
    out = run_orca(name, inp, folder=folder, rerun=rerun)
    if not os.path.isfile(esp):
        lines = open(dens).read().splitlines()
        natoms = abs(int(lines[2].split()[0]))
        origin, axes, rho = read_cube(dens)
        points = origin + np.indices(rho.shape).reshape(3, -1).T @ axes
        np.savetxt(os.path.join(workdir, 'points.txt'), points, header=str(len(points)), comments='')
        subprocess.run([os.path.join(ORCA_DIR, 'orca_vpot'), name + '.gbw', name + '.scfp', 'points.txt', 'esp.txt'],
                       cwd=workdir, capture_output=True)
        v = np.loadtxt(os.path.join(workdir, 'esp.txt'), skiprows=1)[:, 3]
        body = '\n'.join(' '.join(f'{x:.5e}' for x in v[k:k + 6]) for k in range(0, len(v), 6))
        with open(esp, 'w') as f:
            f.write('\n'.join(lines[:6 + natoms]) + '\n' + body + '\n')
        save_to_drive(workdir)
    return out, dens, esp

def show_esp(xyzfile, surfaces, isovalue=0.002, vmax=0.04, width=500, height=400):
    # molecules as sticks, with the ESP (red negative, blue positive, in Eh/e) on the density surface
    view = py3Dmol.view(width=width, height=height)
    view.addModel(open(xyzfile).read(), 'xyz')
    view.setStyle({'stick': {'radius': 0.12}})
    for dens, esp in surfaces:
        view.addVolumetricData(cube_for_display(dens, 32), 'cube',
                               {'isoval': isovalue, 'opacity': 0.75, 'voldata': cube_for_display(esp, 32), 'volformat': 'cube',
                                'volscheme': {'gradient': 'rwb', 'min': -vmax, 'max': vmax}})
    view.zoomTo()
    return view.show()

### All the calculations of a complex

The next cell runs, for each of your complexes:

1.  the interaction energy at the equilibrium distance with HF/def2-SVP, B3LYP-D4/def2-SVP (which also gives B3LYP without dispersion) and r$^2$SCAN-3c, all with the counterpoise correction;
2.  for `BSSE_COMPLEX`, the same with B3LYP-D4 and the larger def2-TZVP basis set, with and without counterpoise correction;
3.  the interaction energy at the five distances of S22x5 (0.9 to 2.0 times the equilibrium distance) with HF/def2-SVP and r$^2$SCAN-3c;
4.  the density, the ESP and the CHELPG charges of the two molecules.

The results are collected in `g1/results/complex_N.json`, read by the report. The cell prints each calculation as it goes; if Colab disconnects, run it again: what is already done is not repeated.

In [ ]:
%%group 1
def coulomb(xyzfile, nA, qA, qB):
    # electrostatic energy (kcal/mol) between the point charges qA (first nA atoms) and qB (the others)
    el, x = read_xyz(xyzfile)
    r = np.linalg.norm(x[:nA, None, :] - x[None, nA:, :], axis=2)
    return 332.06 * np.sum(np.outer(qA, qB) / r)

def run_complex(i):
    xyz, nA, ref = s22_file(i)
    system = s22[s22.id == i].system.iloc[0]
    folder = os.path.join('g1', f'{i:02d}_{system}')
    t0 = time.time()
    res = {'id': i, 'system': system, 'nA': nA, 'reference': ref}
    # 1. equilibrium distance
    hf, _ = interaction_energy_split(xyz, nA, 'HF def2-SVP', folder)
    b3, d4 = interaction_energy_split(xyz, nA, 'B3LYP D4 def2-SVP', folder)
    r2, r2d = interaction_energy_split(xyz, nA, 'r2SCAN-3c', folder)
    res['equilibrium'] = {'HF/SVP': hf, 'B3LYP/SVP': b3, 'B3LYP-D4/SVP': b3 + d4, 'D4 (B3LYP)': d4,
                          'r2SCAN-3c no D4': r2, 'r2SCAN-3c': r2 + r2d}
    # 2. basis set and BSSE
    if i == BSSE_COMPLEX:
        res['bsse'] = {}
        for basis in ['def2-SVP', 'def2-TZVP']:
            no_cp = interaction_energy(xyz, nA, f'B3LYP D4 {basis}', folder, counterpoise=False)
            cp = sum(interaction_energy_split(xyz, nA, f'B3LYP D4 {basis}', folder))
            res['bsse'][basis] = {'no CP': no_cp, 'CP': cp}
    # 3. the S22x5 curve
    res['curve'] = {'scale': SCALES, 'HF/SVP': [], 'r2SCAN-3c no D4': [], 'r2SCAN-3c': [], 'reference': []}
    for s in SCALES:
        f, _, r = s22_file(i, s)
        res['curve']['HF/SVP'].append(interaction_energy_split(f, nA, 'HF def2-SVP', folder)[0])
        e, d = interaction_energy_split(f, nA, 'r2SCAN-3c', folder)
        res['curve']['r2SCAN-3c no D4'].append(e)
        res['curve']['r2SCAN-3c'].append(e + d)
        res['curve']['reference'].append(r)
    # 4. charges and ESP of the two molecules
    lines = xyz_block(xyz).splitlines()
    q = {}
    for k, coords in [('A', lines[:nA]), ('B', lines[nA:])]:
        out, dens, esp = esp_cubes(f'monomer_{k}', '\n'.join(coords), 0, folder)
        q[k] = [c for _, c in charges(out, 'CHELPG')]
        res[f'dipole {k}'] = dipole(out)
    res['charges'] = q
    res['curve']['Coulomb (CHELPG)'] = [coulomb(s22_file(i, s)[0], nA, q['A'], q['B']) for s in SCALES]
    save_json(res, os.path.join('g1', 'results', f'complex_{i}.json'))
    print(f'complex {i} done in {(time.time() - t0) / 60:.1f} minutes\n')

for i in my_complexes:
    run_complex(i)

A first look at your results (the pictures are in the report, section 3.1):

In [ ]:
%%group 1
for i in my_complexes:
    res = json.load(open(os.path.join('g1', 'results', f'complex_{i}.json')))
    print(f"S22 n. {i} {res['system']}: reference {res['reference']:.2f} kcal/mol")
    for method, e in res['equilibrium'].items():
        print(f'   {method:16s} {e:7.2f}')

# GROUP 1, Part 2: the report

The cells of the report read the results of all the members of the group (run them again when the others have finished: `sync_from_drive()` brings their results here).

## 1. Introduction

*Explain to someone who has not studied this topic: what noncovalent interactions are, why they matter for the structure and function of biomolecules (give examples from DNA, RNA and proteins), and what kinds there are. Which complexes did your group study, and which biological interaction does each of them model? Why are quantum calculations useful here, given that MD simulations use force fields?*

*Your text here.*

## 2. Methods

*Describe in a few sentences, without equations, what you computed and how: the S22 complexes and the reference values; how an interaction energy is computed; the methods (HF, B3LYP, the dispersion correction D4, r$^2$SCAN-3c) and what each of them includes or misses; the basis sets and the counterpoise correction; the curves at different distances; the electrostatic potential and the CHELPG charges.*

*Your text here.*

## 3. Results

In [ ]:
%%group 1
sync_from_drive()
results = {}
for f in sorted(glob.glob(os.path.join('g1', 'results', 'complex_*.json'))):
    res = json.load(open(f))
    results[res['id']] = res
print('complexes of the group:', {i: r['system'] for i, r in results.items()})

### 3.1 The complexes and their electrostatic potential

**Figure 1** (one per complex, screenshot): each complex with the ESP of the two molecules. Rotate it to show the contact, and take a screenshot. To see one molecule at a time, keep only one element of the list `surfaces`.

*Describe the figures: where are the red and blue regions of each molecule? In the hydrogen-bonded complexes, which regions face each other? And in the stacked ones? What does this suggest about the importance of electrostatics in the two types of complexes?*

In [ ]:
%%group 1
for i, res in results.items():
    xyz, nA, _ = s22_file(i)
    folder = os.path.join('g1', f"{i:02d}_{res['system']}")
    surfaces = [(f'{folder}/monomer_{k}/monomer_{k}.dens.cube', f'{folder}/monomer_{k}/monomer_{k}.esp.cube') for k in 'AB']
    print(f"S22 n. {i}: {res['system']}")
    show_esp(xyz, surfaces)

*Your text here.*

### 3.2 How strong are the interactions, and how good are the methods?

**Figure 2**: the interaction energies of all the complexes of the group with the different methods, compared with the reference.

*Which complexes are the most strongly bound? Compare H-bonded and stacked complexes of the same molecules. Which methods are within 1 kcal/mol of the reference (a typical target, "chemical accuracy")? For which complexes does HF fail, and why?*

In [ ]:
%%group 1
methods = ['HF/SVP', 'B3LYP/SVP', 'B3LYP-D4/SVP', 'r2SCAN-3c']
table = pd.DataFrame({f"{i} {r['system']}": {**{m: r['equilibrium'][m] for m in methods}, 'reference': r['reference']}
                      for i, r in results.items()}).T
display(table.round(2))

fig, ax = plt.subplots(figsize=(1.6 * len(table) + 3, 4.5))
x = np.arange(len(table))
w = 0.8 / len(methods)
for k, m in enumerate(methods):
    ax.bar(x + (k - (len(methods) - 1) / 2) * w, table[m], width=w * 0.95, label=m)
ax.scatter(x, table['reference'], marker='_', s=900, color='black', zorder=3, label='reference CCSD(T)')
ax.set_xticks(x); ax.set_xticklabels([n.replace('_', ' ') for n in table.index], rotation=25, ha='right')
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('interaction energy (kcal/mol)'); ax.legend(ncol=3, fontsize=9)
plt.tight_layout(); save_figure(fig, 'g1_interaction_energies'); plt.show()

*Your text here.*

### 3.3 Where does the binding come from?

The dispersion correction D4 is a separate term, so we can split the B3LYP-D4 interaction energy in two parts: everything else (electrostatics, polarisation and Pauli repulsion, as described by B3LYP), and dispersion.

**Figure 3**: the two contributions for all the complexes of the group.

*Which complexes are held together mainly by dispersion? Is dispersion negligible for the H-bonded ones? Relate the result to the ESP pictures and to the shape of the molecules (flat aromatic rings vs polar groups).*

In [ ]:
%%group 1
names = [f"{i} {r['system']}".replace('_', ' ') for i, r in results.items()]
elec = np.array([r['equilibrium']['B3LYP/SVP'] for r in results.values()])
disp = np.array([r['equilibrium']['D4 (B3LYP)'] for r in results.values()])
fig, ax = plt.subplots(figsize=(1.4 * len(names) + 3, 4.5))
x = np.arange(len(names))
ax.bar(x, elec, width=0.6, label='B3LYP: electrostatics, polarisation, repulsion')
ax.bar(x, disp, width=0.6, bottom=np.minimum(elec, 0), label='dispersion (D4)')
ax.scatter(x, [r['reference'] for r in results.values()], marker='_', s=900, color='black', zorder=3, label='reference')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=25, ha='right')
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('interaction energy (kcal/mol)'); ax.legend(fontsize=9)
plt.tight_layout(); save_figure(fig, 'g1_dispersion'); plt.show()
for n, e, d in zip(names, elec, disp):
    print(f'{n:45s} dispersion = {100 * d / (e + d):5.0f}% of the B3LYP-D4 interaction energy')

*Your text here.*

### 3.4 The basis set and the basis-set superposition error

*Report the B3LYP-D4 interaction energies with def2-SVP and def2-TZVP, with and without the counterpoise correction, for the complexes for which you computed them. How large is the BSSE, in absolute value and as a fraction of the interaction energy? Is the uncorrected small-basis result closer to the reference? (Sometimes errors cancel: a good result for the wrong reason.)*

In [ ]:
%%group 1
for i, r in results.items():
    if 'bsse' in r:
        print(f"S22 n. {i} {r['system']} (reference {r['reference']:.2f} kcal/mol)")
        for basis, v in r['bsse'].items():
            print(f"   {basis:10s} no CP {v['no CP']:7.2f}   CP {v['CP']:7.2f}   BSSE {v['CP'] - v['no CP']:5.2f} kcal/mol")

*Your text here.*

### 3.5 Pulling the molecules apart

**Figure 4**: the interaction energy as a function of the distance between the two molecules (relative to the equilibrium distance), computed with HF, with r$^2$SCAN-3c with and without its dispersion term, and with a simple point-charge model: the Coulomb energy between the CHELPG charges of the two molecules, which is what a force field uses for electrostatics.

*Which methods find the minimum at the right distance? Are the stacked complexes bound at all without dispersion? How quickly do the different contributions decay with the distance? Where does the point-charge model work, and where does it fail (think of what happens when the electron clouds overlap)?*

In [ ]:
%%group 1
n = len(results)
fig, axs = plt.subplots(1, n, figsize=(4.5 * n, 4), squeeze=False)
styles = {'reference': 'ko-', 'HF/SVP': 's--', 'r2SCAN-3c no D4': '^--', 'r2SCAN-3c': 'o-', 'Coulomb (CHELPG)': 'd:'}
for ax, (i, r) in zip(axs[0], results.items()):
    for key, st in styles.items():
        ax.plot(r['curve']['scale'], r['curve'][key], st, label=key, ms=5)
    ax.axhline(0, c='grey', lw=0.6)
    ax.set_title(f"{i} {r['system']}".replace('_', ' '), fontsize=10)
    ax.set_xlabel('distance / equilibrium distance')
axs[0, 0].set_ylabel('interaction energy (kcal/mol)'); axs[0, 0].legend(fontsize=8)
plt.tight_layout(); save_figure(fig, 'g1_curves'); plt.show()

*Your text here.*

### 3.6 Back to the real biomolecules

The S22 complexes are small models. The cells below show where the same interactions occur in real structures: a DNA double helix (PDB 1BNA), a transfer RNA (1EHZ), and a protein kinase (the Abl kinase of the docking practical, 2GQG). The DNA and RNA structures are downloaded by your browser from the PDB; if they don't appear, download the file from [rcsb.org](https://www.rcsb.org) and open it in VMD or ChimeraX.

**Figure 5** (screenshot): your interactions in their biological context.

*Compare the arrangement in the real structure with the S22 complex (distances, orientation: you can measure them in VMD or ChimeraX). Estimate how much these interactions contribute to the stability of the structure (e.g. how many base pairs and stacking contacts in one turn of DNA, how many aromatic contacts in the core of the kinase). What is different in the real system (water, neighbouring groups, the backbone that holds the bases)?*

In [ ]:
%%group 1
KINASE = os.path.join(REPO_DIR, 'Data', 'docking', '2gqg.pdb')

def backbone_hbonds(pdbfile, cutoff=3.1):
    # backbone N...O pairs closer than cutoff (Å), between residues at least 3 apart: the H-bonds of helices and sheets
    N, O = {}, {}
    for l in open(pdbfile):
        if l.startswith('ATOM') and l[12:16].strip() in ('N', 'O'):
            (N if l[12:16].strip() == 'N' else O)[int(l[22:26])] = np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
    return [(N[i], O[j]) for i in N for j in O if abs(i - j) > 2 and np.linalg.norm(N[i] - O[j]) < cutoff]

trp_tyr = dict(local_file=KINASE, highlights=[{'resi': [261, 264]}, {'resi': [478, 449]}], labels=[{'resi': [261, 264, 478, 449]}],
               title='Abl kinase (2GQG): Trp261 stacked on Tyr264, and Trp478 with Tyr449')
uridines = dict(pdb='1EHZ', highlights=[{'resn': 'U'}],
                title='tRNA (1EHZ): all the uridines. Find one stacked on another base, and one in a base pair')
CONTEXT = {
    7:  dict(pdb='1BNA', highlights=[{'chain': 'A', 'resi': 6}, {'chain': 'B', 'resi': 19}],
             title='DNA (1BNA): the Watson-Crick pair A6-T19'),
    15: dict(pdb='1BNA', highlights=[{'chain': 'A', 'resi': 5}, {'chain': 'A', 'resi': 6}],
             title='DNA (1BNA): A5 stacked on A6 along the same strand'),
    5: uridines, 13: uridines, 14: trp_tyr, 21: trp_tyr,
    4:  dict(local_file=KINASE, highlights=[], lines=backbone_hbonds(KINASE),
             title='Abl kinase (2GQG): the hydrogen bonds of the backbone (orange dashed lines) in helices and sheets'),
    3:  dict(local_file=KINASE, highlights=[{'resi': [363, 381]}], labels=[{'resi': [363, 381]}],
             title='Abl kinase (2GQG): the carboxylic groups of Asp363 and Asp381 at 2.9 Å (one of them must be protonated)'),
    22: dict(local_file=KINASE, highlights=[{'resi': [320, 326]}], labels=[{'resi': [320, 326]}],
             title='Abl kinase (2GQG): Tyr320 and Tyr326'),
    17: dict(local_file=KINASE, highlights=[{'resi': [410, 416]}], labels=[{'resi': [410, 416]}],
             title='Abl kinase (2GQG): the O-H group of Ser410 pointing to the ring of Phe416'),
}
shown = []
for i in results:
    c = CONTEXT.get(i)
    if c is None:
        print(f'S22 n. {i}: find yourself an example in a structure of the PDB')
    elif c not in shown:
        shown.append(c)
        show_pdb_context(c.get('pdb', ''), c['highlights'], title=c['title'], labels=c.get('labels', ()),
                         lines=c.get('lines', ()), local_file=c.get('local_file'))

*Your text here.*

## 4. Discussion

*Interpret the results of the whole group. Some points to consider:*

*   *Rank the interactions you studied by strength. Is base pairing or base stacking stronger? In water the bases can also make hydrogen bonds with the solvent: what would you expect to change? What does this suggest about what really stabilises the double helix?*
*   *Which methods would you trust to study a new complex, and which not? What is the price (computer time)?*
*   *A force field describes these interactions with fixed point charges and a Lennard-Jones term. From your results (the point-charge curves, the role of dispersion, the role of polarisation), which interactions can it reproduce well, and which are more difficult?*

## 5. Conclusions

## References

*   Jurečka P et al. (2006) Benchmark database of accurate (MP2 and CCSD(T) complete basis set limit) interaction energies of small model complexes, DNA base pairs, and amino acid pairs. *Phys Chem Chem Phys* 8:1985
*   Takatani T et al. (2010) Basis set consistent revision of the S22 test set of noncovalent interaction energies. *J Chem Phys* 132:144104
*   Gráfová L et al. (2010) Comparative study of selected wave function and density functional methods for noncovalent interaction energy calculations using the extended S22 data set. *J Chem Theory Comput* 6:2365
*   Grimme S et al. (2021) r$^2$SCAN-3c: a "Swiss army knife" composite electronic-structure method. *J Chem Phys* 154:064103
*   *add the papers you used*

## 6. Going further (optional)

Two possible extensions, in different directions: choose one (or both, or one per member), or design your own. Add the results to the report and to the discussion.

### 6.1 In water: does the solvent weaken the interactions?

All our energies are in vacuum, but biomolecules live in water. Water competes for the hydrogen bonds (a base can make hydrogen bonds with water instead of with its partner), and its high dielectric constant screens the electrostatic interactions. Here we recompute the interaction energies with r$^2$SCAN-3c in vacuum and in water described as a continuum (CPCM, as group 2 does for the spectra). There is no counterpoise correction here: r$^2$SCAN-3c already contains a correction for the BSSE, and ghost atoms don't mix well with the continuum. **Time**: about 10 minutes for a pair of complexes of 24 atoms, 20 for adenine–thymine. You can add the complexes of the other members to `SOLVENT_COMPLEXES`.

*Which interactions lose more in water, hydrogen bonds or stacking? What does this suggest about the role of base stacking in the stability of DNA? What is still missing in a continuum model? (Think about the hydrophobic effect: the water molecules released when two flat surfaces stick together.)*

In [ ]:
%%group 1
SOLVENT_COMPLEXES = my_complexes

def interaction_energy_solvent(i, solvent=None):
    # r2SCAN-3c interaction energy (kcal/mol) in vacuum (solvent=None) or in a continuum solvent
    xyz, nA, ref = s22_file(i)
    folder = os.path.join('g1', f"{i:02d}_{s22[s22.id == i].system.iloc[0]}", 'solvent')
    lines = xyz_block(xyz).splitlines()
    keywords = '! r2SCAN-3c TightSCF' + (f' CPCM({solvent})' if solvent else '')
    E = {}
    for k, coords in [('AB', lines), ('A', lines[:nA]), ('B', lines[nA:])]:
        E[k] = final_energy(run_orca(f'{k}_{solvent or "vacuum"}', keywords + '\n* xyz 0 1\n' + '\n'.join(coords) + '\n*', folder=folder))
    return (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL

rows = []
for i in SOLVENT_COMPLEXES:
    rows.append({'complex': f"{i} {s22[s22.id == i].system.iloc[0]}".replace('_', ' '),
                 'vacuum': interaction_energy_solvent(i), 'water (CPCM)': interaction_energy_solvent(i, 'water')})
solv = pd.DataFrame(rows).set_index('complex')
solv['fraction left in water'] = solv['water (CPCM)'] / solv['vacuum']
display(solv.round(2))
fig, ax = plt.subplots(figsize=(1.5 * len(solv) + 3, 4))
solv[['vacuum', 'water (CPCM)']].plot.bar(ax=ax, rot=20)
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('interaction energy (kcal/mol)'); ax.set_xlabel('')
plt.tight_layout(); save_figure(fig, 'g1_water'); plt.show()

*Your text here.*

### 6.2 A different interaction: the cation–π interaction

The S22 set has no charged molecules, but proteins are full of them. A positively charged side chain (lysine, arginine) on top of an aromatic ring (Phe, Tyr, Trp) makes a surprisingly strong **cation–π interaction**. It holds acetylcholine in the "aromatic box" of its receptor and of acetylcholinesterase, and it lets reader proteins recognise the methylated lysines of histones. Here we model it with an ammonium ion NH$_4^+$ (the end of a lysine side chain) above the centre of a benzene ring (Phe), with one N–H pointing to the ring, and compute the interaction energy with r$^2$SCAN-3c as a function of the distance between N and the ring. As in section 3.5, we also compute the Coulomb energy between the CHELPG charges of the two molecules. **Time**: about 3 minutes.

*How strong is this interaction, compared with the hydrogen bonds and the stacking of the S22 complexes? Benzene has no net charge: why does it attract a cation so strongly? (Look at the ESP of benzene, negative above and below the ring: you can compute it with `esp_cubes`.) How much of the interaction do the point charges capture, and what is missing? Optional: repeat the calculation in water (CPCM, see 6.1), or replace NH$_4^+$ with a water molecule or a Na$^+$ ion.*

In [ ]:
%%group 1
CATION_DISTANCES = [2.6, 2.9, 3.2, 3.6, 4.2, 5.0, 6.0]      # distance (Å) between N and the centre of the ring

def benzene():
    atoms = []
    for k in range(6):
        a = np.radians(60 * k)
        atoms += [('C', 1.39 * np.cos(a), 1.39 * np.sin(a), 0.0), ('H', 2.47 * np.cos(a), 2.47 * np.sin(a), 0.0)]
    return atoms

def ammonium(z):
    # NH4+ (tetrahedral, N-H 1.03 Å) with N on the axis of the ring at height z and one N-H pointing to the ring
    t = np.array([[0, 0, -1.0], [0.9428, 0, 0.3333], [-0.4714, 0.8165, 0.3333], [-0.4714, -0.8165, 0.3333]])
    return [('N', 0.0, 0.0, z)] + [('H', 1.03 * x, 1.03 * y, z + 1.03 * w) for x, y, w in t]

def block(atoms):
    return '\n'.join(f'{e} {x:10.4f} {y:10.4f} {z:10.4f}' for e, x, y, z in atoms)

folder = os.path.join('g1', 'cation_pi')
out_b = run_orca('benzene', '! r2SCAN-3c CHELPG TightSCF\n* xyz 0 1\n' + block(benzene()) + '\n*', folder=folder)
qb = [c for _, c in charges(out_b, 'CHELPG')]
xb = np.array([a[1:] for a in benzene()])
energies, coulombs = [], []
for z in CATION_DISTANCES:
    out_n = run_orca(f'ammonium_{z}', '! r2SCAN-3c CHELPG TightSCF\n* xyz 1 1\n' + block(ammonium(z)) + '\n*', folder=folder)
    out_c = run_orca(f'complex_{z}', '! r2SCAN-3c TightSCF\n* xyz 1 1\n' + block(benzene() + ammonium(z)) + '\n*', folder=folder)
    energies.append((final_energy(out_c) - final_energy(out_b) - final_energy(out_n)) * HARTREE2KCAL)
    qn = [c for _, c in charges(out_n, 'CHELPG')]
    xn = np.array([a[1:] for a in ammonium(z)])
    coulombs.append(332.06 * np.sum(np.outer(qb, qn) / np.linalg.norm(xb[:, None] - xn[None], axis=2)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(CATION_DISTANCES, energies, 'o-', label='r$^2$SCAN-3c')
ax.plot(CATION_DISTANCES, coulombs, 'd:', label='Coulomb (CHELPG charges)')
ax.axhline(0, color='grey', lw=0.8); ax.set_xlabel('distance N - centre of the ring (Å)'); ax.set_ylabel('interaction energy (kcal/mol)')
ax.legend(); ax.set_title('NH$_4^+$ on benzene', loc='left')
plt.tight_layout(); save_figure(fig, 'g1_cation_pi'); plt.show()
best = CATION_DISTANCES[int(np.argmin(energies))]
print(f'minimum: {min(energies):.1f} kcal/mol at {best} Å')
with open(os.path.join(folder, 'complex_min.xyz'), 'w') as f:
    f.write(f'17\nNH4+ on benzene at {best} A\n' + block(benzene() + ammonium(best)) + '\n')
show_molecule(os.path.join(folder, 'complex_min.xyz'))

*Your text here.*

# GROUP 2, Part 1: running the calculations

Write the names of your molecules (the files in `Data/qm/chromophores`, without `.xyz`) and run the cells below.

*   `FUNC_MOLECULES`: the molecules for which the excitation energies are also computed with three other functionals. For the large molecules (retinal, lumiflavin) one is enough.
*   `FMIN`: the oscillator strength above which a state is considered *bright*. The first absorption band of most chromophores is strong ($f$ > 0.1); the bands of the aromatic side chains at 260–280 nm are very weak ($f$ ≈ 0.001–0.01): member 5 should use `FMIN = 0.001`.

In [ ]:
%%group 2
my_molecules = ['hbdi_neutral', 'hbdi_anion', 'hbdi_cation']
FUNC_MOLECULES = my_molecules            # e.g. my_molecules[:1] for retinal and lumiflavin
FMIN = 0.01                              # 0.001 for the aromatic side chains
NROOTS = 10                              # number of excited states

CHROMO_DIR = os.path.join(QM_DIR, 'chromophores')
def charge_of(name):
    return int(re.search(r'charge=([+-]?\d+)', open(os.path.join(CHROMO_DIR, name + '.xyz')).readlines()[1]).group(1))

def lowest_bright(states, fmin=None):
    # the lowest excited state with oscillator strength above fmin: the first absorption band
    states = np.array(states)
    return states[np.argmax(states[:, 2] > (FMIN if fmin is None else fmin))]

for m in my_molecules:
    print(m, 'charge', charge_of(m))
    show_molecule(os.path.join(CHROMO_DIR, m + '.xyz'))

### All the calculations of a molecule

The next cell runs, for each of your molecules:

1.  the optimisation of the structure with GFN2-xTB (seconds);
2.  TD-DFT with CAM-B3LYP/def2-SVP in vacuum, which also writes the HOMO and the LUMO in cube files;
3.  the same in water, described as a dielectric continuum (CPCM): the molecule sits in a cavity surrounded by a polarisable medium;
4.  for the molecules in `FUNC_MOLECULES`, TD-DFT in vacuum with three other functionals: PBE0 and B3LYP (hybrids, with 25% and 20% of Hartree–Fock exchange), and ωB97X-D3 (range-separated, like CAM-B3LYP).

The results are collected in `g2/results/<molecule>.json`. **Time** (one TD-DFT calculation): about 1 minute for the aromatic side chains, 5 for HBDI and lumiflavin, 15 for retinal. If Colab disconnects, run the cell again.

In [ ]:
%%group 2
def n_electrons(xyzfile, charge):
    Z = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'S': 16}
    el, _ = read_xyz(xyzfile)
    return sum(Z[e] for e in el) - charge

def tddft(name, xyzfile, charge, folder, method='CAM-B3LYP def2-SVP', solvent=None, cubes=False, nroots=None, extra=''):
    homo = n_electrons(xyzfile, charge) // 2 - 1
    keywords = f'! {method} TightSCF' + (f' CPCM({solvent})' if solvent else '')
    plots = plots_block([f'MO("{name}.homo.cube",{homo},0)', f'MO("{name}.lumo.cube",{homo + 1},0)']) if cubes else ''
    inp = f'{keywords}\n{extra}%tddft\n  NRoots {nroots or NROOTS}\nend\n{plots}* xyz {charge} 1\n{xyz_block(xyzfile)}\n*'
    return run_orca(name, inp, folder=folder)

def run_molecule(m):
    folder = os.path.join('g2', m)
    t0 = time.time()
    q = charge_of(m)
    opt = run_orca(m + '_opt', f'! Native-GFN2-XTB Opt\n* xyz {q} 1\n{xyz_block(os.path.join(CHROMO_DIR, m + ".xyz"))}\n*', folder=folder)
    xyz = os.path.join(os.path.dirname(opt), m + '_opt.xyz')
    res = {'molecule': m, 'charge': q, 'states': {}}
    out = tddft(m + '_cam_gas', xyz, q, folder, cubes=True)
    res['states']['CAM-B3LYP vacuum'] = excitations(out).tolist()
    out = tddft(m + '_cam_water', xyz, q, folder, solvent='water')
    res['states']['CAM-B3LYP water'] = excitations(out).tolist()
    if m in FUNC_MOLECULES:
        for func in ['PBE0', 'B3LYP', 'wB97X-D3']:
            out = tddft(f'{m}_{func}_gas', xyz, q, folder, method=f'{func} def2-SVP')
            res['states'][f'{func} vacuum'] = excitations(out).tolist()
    save_json(res, os.path.join('g2', 'results', m + '.json'))
    print(f'{m} done in {(time.time() - t0) / 60:.1f} minutes\n')

for m in my_molecules:
    run_molecule(m)

A first look at your results: the excited states in vacuum and the orbitals involved (e.g. `40a -> 41a` is HOMO → LUMO if the HOMO is orbital 40; the number of the HOMO is printed).

In [ ]:
%%group 2
for m in my_molecules:
    res = json.load(open(os.path.join('g2', 'results', m + '.json')))
    xyz = os.path.join('g2', m, m + '_opt', m + '_opt.xyz')
    print(f'{m}: the HOMO is orbital {n_electrons(xyz, res["charge"]) // 2 - 1}')
    transitions(os.path.join('g2', m, m + '_cam_gas', m + '_cam_gas.out'))
    e, lam, f = lowest_bright(res['states']['CAM-B3LYP vacuum'])
    print(f'   first bright state: {e:.2f} eV, {lam:.0f} nm, f = {f:.3f}\n')

# GROUP 2, Part 2: the report

The cells of the report read the results of all the members of the group (run them again when the others have finished: `sync_from_drive()` brings their results here).

## 1. Introduction

*Explain to someone who has not studied this topic: why most biomolecules are colourless, what makes a molecule coloured, and why colour matters in biology (vision, fluorescent proteins as tools, photosynthesis, enzymes that change colour). Which chromophores did your group study, where are they found, and what question did each member ask?*

*Your text here.*

## 2. Methods

*Describe in a few sentences, without equations, what you computed and how: how the structures were obtained; what TD-DFT computes (excitation energies, oscillator strengths) and how a spectrum and a colour are obtained from them; the functionals and why we compare several of them; how the solvent was described.*

*Your text here.*

## 3. Results

In [ ]:
%%group 2
sync_from_drive()
results = {}
for f in sorted(glob.glob(os.path.join('g2', 'results', '*.json'))):
    res = json.load(open(f))
    results[res['molecule']] = res
print('molecules of the group:', list(results))

### 3.1 Where is the electron going?

In the brightest excitations an electron goes (mostly) from the HOMO to the LUMO. Comparing the two orbitals shows which part of the molecule absorbs the light, and whether the excitation moves charge from one part of the molecule to another (a *charge-transfer* excitation, very sensitive to the environment).

**Figure 1** (screenshot, one per molecule): HOMO (left) and LUMO (right) of each molecule. The two views are linked: rotating one rotates the other.

*Which atoms contribute to the HOMO and the LUMO? Is the excitation spread over the whole conjugated system? Is charge displaced? Compare the molecules of your system (e.g. neutral vs anion, short vs long chain, oxidised vs reduced).*

In [ ]:
%%group 2
def show_homo_lumo(m, isovalue=0.03, width=760, height=300):
    folder = os.path.join('g2', m, m + '_cam_gas')
    view = py3Dmol.view(viewergrid=(1, 2), width=width, height=height, linked=True)
    for col, orb in enumerate(['homo', 'lumo']):
        cube = os.path.join(folder, f'{m}_cam_gas.{orb}.cube')
        view.addModel(cube_atoms_xyz(cube), 'xyz', viewer=(0, col))
        data = cube_for_display(cube)
        view.addVolumetricData(data, 'cube', {'isoval': isovalue, 'color': 'red', 'opacity': 0.8}, viewer=(0, col))
        view.addVolumetricData(data, 'cube', {'isoval': -isovalue, 'color': 'blue', 'opacity': 0.8}, viewer=(0, col))
    view.setStyle({'stick': {'radius': 0.12}})
    view.zoomTo()
    return view.show()

for m in results:
    print(f'{m}: HOMO (left) and LUMO (right)')
    show_homo_lumo(m)

*Your text here.*

### 3.2 Spectra and colours

**Figure 2**: the computed spectra (CAM-B3LYP, in water) of all the molecules of the group, with the visible region in colour. **Figure 3**: the colour predicted for a solution of each molecule (as at the end of the training).

*Which molecules absorb in the visible? Which colour do the calculations predict, and which colour do the real molecules have? How do the spectra change within your system (protonation, length of the chain, oxidation state)?*

In [ ]:
%%group 2
condition = 'CAM-B3LYP water'
emin, emax = 1.5, 7.0
fig, ax = plt.subplots(figsize=(9, 4.5))
for lam in np.arange(380, 750, 2):              # the visible region, coloured
    ax.axvspan(1239.84 / (lam + 1), 1239.84 / (lam - 1), color=plt.cm.nipy_spectral(0.05 + 0.85 * (lam - 380) / 370), alpha=0.18, lw=0)
for m, r in results.items():
    st = np.array(r['states'][condition])
    E, eps = spectrum(st, emin=emin, emax=emax)
    line, = ax.plot(E, eps, label=m)
    ax.vlines(st[:, 0], 0, st[:, 2] * 3, colors=line.get_color(), lw=1)
ax.set_xlim(emin, emax); ax.set_xlabel('excitation energy (eV)'); ax.set_ylabel('absorption (arb. units)')
secax = ax.secondary_xaxis('top', functions=(lambda e: 1239.84 / np.maximum(e, 1e-3), lambda l: 1239.84 / np.maximum(l, 1e-3)))
secax.set_xlabel('wavelength (nm)'); ax.legend(fontsize=9); ax.set_title(condition, loc='left')
plt.tight_layout(); save_figure(fig, 'g2_spectra'); plt.show()

fig, ax = plt.subplots(figsize=(2.2 * len(results), 2))
for k, (m, r) in enumerate(results.items()):
    ax.add_patch(plt.Rectangle((k * 2.2, 0), 1.0, 1.0, color=colour_of(np.array(r['states'][condition])), ec='grey'))
    ax.text(k * 2.2 + 0.5, -0.1, m, ha='center', va='top', fontsize=9)
ax.set_xlim(-0.6, 2.2 * len(results)); ax.set_ylim(-0.6, 1.1); ax.axis('off')
save_figure(fig, 'g2_colours'); plt.show()

*Your text here.*

### 3.3 Comparison with experiment

Write in the dictionary below the experimental wavelength of the first absorption band of each molecule, with its reference in the report (`None` if it is not known). The values given are approximate: look for the original papers.

**Figure 4**: computed vs experimental wavelength. Points on the diagonal are perfect predictions.

*How large are the errors, in nm and in eV? Is the error systematic (always too short or always too long wavelengths)? Is the trend among your molecules correct even when the absolute values are not?*

In [ ]:
%%group 2
EXPERIMENT = {          # nm, first absorption band (check the values and cite the papers in the report)
    'hbdi_neutral': 370, 'hbdi_anion': 425, 'hbdi_cation': 395,        # in water
    'retinal_psb_alltrans': 440, 'retinal_psb_11cis': None,            # in methanol
    'psb_2': None, 'psb_3': None, 'psb_4': None, 'psb_5': None, 'psb_6': None,
    'lumiflavin_ox': 445, 'lumiflavin_red': None,                      # in water
    'toluene': 262, 'p_cresol': 277, 'p_cresolate': 295, '3_methylindole': 280,
}
rows = []
for m, r in results.items():
    row = {'molecule': m}
    for cond in ['CAM-B3LYP vacuum', 'CAM-B3LYP water']:
        e, lam, f = lowest_bright(r['states'][cond])
        row[cond + ' (nm)'] = round(lam)
        row[cond + ' f'] = round(f, 3)
    row['experiment (nm)'] = EXPERIMENT.get(m)
    rows.append(row)
comparison = pd.DataFrame(rows).set_index('molecule')
display(comparison)

known = comparison[comparison['experiment (nm)'].notna()]
if len(known):
    fig, ax = plt.subplots(figsize=(5, 5))
    for cond, mk in [('CAM-B3LYP vacuum', 'o'), ('CAM-B3LYP water', 's')]:
        ax.scatter(known['experiment (nm)'], known[cond + ' (nm)'], marker=mk, s=50, label=cond)
    for m, row in known.iterrows():
        ax.annotate(m, (row['experiment (nm)'], row['CAM-B3LYP water (nm)']), fontsize=8, xytext=(4, -10), textcoords='offset points')
    lo = min(known['experiment (nm)'].min(), known['CAM-B3LYP vacuum (nm)'].min()) - 30
    hi = max(known['experiment (nm)'].max(), known['CAM-B3LYP water (nm)'].max()) + 30
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel('experiment (nm)'); ax.set_ylabel('computed (nm)'); ax.legend(fontsize=9)
    plt.tight_layout(); save_figure(fig, 'g2_experiment'); plt.show()

*Your text here.*

### 3.4 How much does the result depend on the functional?

**Figure 5**: the wavelength of the first bright state with four functionals (in vacuum), and the experimental value (in solution).

*How large is the spread among functionals (in nm and in eV)? Is the ordering of the molecules the same with all of them? Which functional is closest to experiment? What does this tell you about the reliability of a single TD-DFT calculation?*

In [ ]:
%%group 2
funcs = ['PBE0', 'B3LYP', 'CAM-B3LYP', 'wB97X-D3']
fig, ax = plt.subplots(figsize=(7, 0.6 * len(results) + 1.5))
for k, (m, r) in enumerate(results.items()):
    for j, func in enumerate(funcs):
        key = f'{func} vacuum'
        if key in r['states']:
            ax.scatter(lowest_bright(r['states'][key])[1], k, color=f'C{j}', s=60, label=func if k == 0 else None)
    if EXPERIMENT.get(m):
        ax.scatter(EXPERIMENT[m], k, marker='|', s=400, color='black', label='experiment (solution)' if k == 0 else None)
ax.set_yticks(range(len(results))); ax.set_yticklabels(list(results))
ax.set_xlabel('wavelength of the first bright state (nm)'); ax.legend(fontsize=8, loc='best')
plt.tight_layout(); save_figure(fig, 'g2_functionals'); plt.show()

*Your text here.*

### 3.5 The effect of water

*From the table of section 3.3: does water shift the absorption to longer (red shift) or shorter (blue shift) wavelengths? Is the shift the same for neutral and charged molecules? A continuum model does not describe specific interactions such as hydrogen bonds between the solvent and the chromophore: could they matter for your molecules?*

*Your text here.*

### 3.6 The chromophore in its protein

The cells below show the chromophores inside their proteins (the structures are downloaded by your browser from the PDB; if they don't appear, download them from [rcsb.org](https://www.rcsb.org) and open them in VMD or ChimeraX):

*   GFP (PDB 1EMA): the chromophore (residue 66, formed by the protein itself from residues 65–67) in the middle of a β-barrel, with residues that make hydrogen bonds with it (His148, Thr203, Ser205, Glu222);
*   rhodopsin (1U19): 11-*cis* retinal bound to Lys296 as a protonated Schiff base, with its counterion Glu113; bacteriorhodopsin (1C3W): all-*trans* retinal bound to Lys216, counterion Asp85;
*   DNA photolyase (1DNP): the flavin (FAD) that repairs DNA damaged by UV light, using the energy of blue light;
*   for the aromatic side chains: the Trp, Tyr and Phe residues of the Abl kinase (2GQG), responsible for its absorption at 280 nm.

**Figure 6** (screenshot): your chromophore in its protein.

*Look at the environment of the chromophore: which groups are close to it (charges, hydrogen bonds, aromatic rings, water)? Is it planar, or twisted by the protein? Using your results (protonation, solvent, conformation), explain how the protein can tune the colour, and answer the question of your system:*

*   *GFP: the neutral form absorbs at 395 nm, but GFP emits green light (≈ 508 nm) from the anionic form: what has to happen in the excited state? (Look up "excited-state proton transfer".)*
*   *Retinal: what is the "opsin shift" (the difference between the absorption in the protein and in methanol) for rhodopsin and bacteriorhodopsin, in eV? What could cause it?*
*   *Conjugated chains: plot the excitation energy vs the number of double bonds N and vs 1/N. In the "particle in a box" model the π electrons move freely along the chain, and the HOMO–LUMO gap decreases as the chain gets longer. Extrapolate to retinal (N = 6) and compare.*
*   *Flavin: which orbitals are involved in the visible band of the oxidised form, and what happens to them in the reduced form? Why is the colour change useful to follow the activity of flavoenzymes?*
*   *Aromatic side chains: which one absorbs most strongly above 250 nm? Why is the absorbance at 280 nm used to measure the concentration of proteins, and why does it depend on the sequence? What happens to the spectrum of Tyr at high pH?*

In [ ]:
%%group 2
KINASE = os.path.join(REPO_DIR, 'Data', 'docking', '2gqg.pdb')
CONTEXT = {
    'hbdi': [dict(pdb='1EMA', highlights=[{'resi': 66}, {'resi': [148, 203, 205, 222]}], labels=[{'resi': [148, 203, 205, 222]}],
                  title='GFP (1EMA): the chromophore and its hydrogen-bond partners')],
    'retinal': [dict(pdb='1U19', highlights=[{'resn': 'RET'}, {'chain': 'A', 'resi': [296, 113]}], labels=[{'chain': 'A', 'resi': [296, 113]}],
                     title='rhodopsin (1U19): 11-cis retinal, Lys296 and Glu113'),
                dict(pdb='1C3W', highlights=[{'resn': 'RET'}, {'resi': [216, 85]}], labels=[{'resi': [216, 85]}],
                     title='bacteriorhodopsin (1C3W): all-trans retinal, Lys216 and Asp85')],
    'lumiflavin': [dict(pdb='1DNP', highlights=[{'resn': 'FAD'}], title='DNA photolyase (1DNP): the flavin (FAD)')],
    'aromatic': [dict(local_file=KINASE, highlights=[{'resn': 'TRP'}, {'resn': 'TYR'}, {'resn': 'PHE'}],
                      title='Abl kinase (2GQG): Trp (green), Tyr (orange) and Phe (cyan)')],
}

def context_key(m):
    if m.startswith('hbdi'):
        return 'hbdi'
    if m.startswith(('retinal', 'psb')):
        return 'retinal'
    if m.startswith('lumiflavin'):
        return 'lumiflavin'
    return 'aromatic'

for key in dict.fromkeys(context_key(m) for m in results):
    for c in CONTEXT[key]:
        show_pdb_context(c.get('pdb', ''), c['highlights'], title=c['title'], labels=c.get('labels', ()), local_file=c.get('local_file'))

*Your text here.*

## 4. Discussion

*Interpret the results of the whole group. Some points to consider:*

*   *Which molecular features move the absorption into the visible (length of the conjugated chain, charges, protonation, charge transfer)?*
*   *How reliable are the calculations: for the absolute colour, and for the trends? What are the main sources of error (functional, basis set, solvent model, structure, vibrations)?*
*   *Proteins tune the colour of their chromophores: summarise the mechanisms that came out of your systems. How would you model a chromophore inside its protein? (Think about QM/MM, lecture 8.)*

## 5. Conclusions

## References

*   Dreuw A, Head-Gordon M (2005) Single-reference ab initio methods for the calculation of excited states of large molecules. *Chem Rev* 105:4009
*   Laurent AD, Jacquemin D (2013) TD-DFT benchmarks: a review. *Int J Quantum Chem* 113:2019
*   Bannwarth C, Ehlert S, Grimme S (2019) GFN2-xTB: an accurate and broadly parametrized self-consistent tight-binding quantum chemical method. *J Chem Theory Comput* 15:1652
*   *add the experimental papers you used (one for each experimental value)*

## 6. Going further (optional)

Two possible extensions, in different directions: the electrostatic effect of the protein, and the motion of the molecule. Choose one (or both, or one per member), or design your own. Add the results to the report and to the discussion. For retinal, these calculations are long: use one of the shorter chains (`psb_4`, `psb_5`) as a model.

### 6.1 The protein as a set of charges: electrostatic colour tuning

Inside a protein, the chromophore feels the electric field of the charged and polar groups around it. A simple way to mimic it is to add **point charges** to the calculation: they have no electrons, but their field polarises the electrons of the chromophore, differently in the ground and in the excited state. In rhodopsin, for example, the positive charge of the protonated Schiff base is balanced by the negative counterion Glu113. Moving the counterion (and other charged or polar groups) is one of the ways in which opsins tune the colour of the same retinal, from blue to red.

The cell places a point charge `PC_CHARGE` at different distances from the atom `PC_ATOM`, outside the molecule (along the line from the centre of the molecule to that atom), and recomputes the excitation energies (5 states, CAM-B3LYP, vacuum). Look at the picture to choose the atom: the numbers are the indices of the atoms. **Time**: about 1 minute per distance for the short chains, 5 for HBDI and lumiflavin.

*Does the charge shift the absorption to longer or shorter wavelengths? Repeat with the charge at the other end of the molecule (another `PC_ATOM`): is the shift the same? Explain the result by thinking about where the charge of the molecule moves in the excited state (compare HOMO and LUMO, section 3.1). How could a protein make the same chromophore absorb blue or red light?*

In [ ]:
%%group 2
PC_MOLECULE = my_molecules[0]
PC_ATOM = None                # index of the atom close to which the charge is placed (None: the first N or O of the molecule)
PC_CHARGE = -1.0              # a counterion; use +1.0 for a positive charge
PC_DISTANCES = [3.0, 4.0, 6.0, 10.0]

xyz = os.path.join('g2', PC_MOLECULE, PC_MOLECULE + '_opt', PC_MOLECULE + '_opt.xyz')
el, x = read_xyz(xyz)
if PC_ATOM is None:
    PC_ATOM = next(i for i, e in enumerate(el) if e in ('N', 'O'))
view = py3Dmol.view(width=500, height=350)
view.addModel(open(xyz).read(), 'xyz')
view.setStyle({'stick': {'radius': 0.12}})
view.addPropertyLabels('index', {'not': {'elem': 'H'}}, {'fontSize': 10, 'showBackground': False, 'fontColor': 'black'})
view.zoomTo()
view.show()

direction = x[PC_ATOM] - x.mean(axis=0)
direction /= np.linalg.norm(direction)
folder = os.path.join('g2', PC_MOLECULE, 'point_charges')
os.makedirs(folder, exist_ok=True)
q = charge_of(PC_MOLECULE)
reference = lowest_bright(excitations(tddft(f'{PC_MOLECULE}_cam_5roots', xyz, q, folder, nroots=5)))
rows = [{'distance (Å)': 'no charge', 'E (eV)': reference[0], 'wavelength (nm)': reference[1], 'f': reference[2]}]
for d in PC_DISTANCES:
    p = x[PC_ATOM] + d * direction
    # the file of the point charges is written outside the folder of the calculation (which is emptied before each run)
    pcfile = os.path.abspath(os.path.join(folder, f'charge_{PC_ATOM}_{PC_CHARGE:+.1f}_{d}.pc'))
    with open(pcfile, 'w') as f:
        f.write(f'1\n{PC_CHARGE} {p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n')
    out = tddft(f'{PC_MOLECULE}_pc_{PC_ATOM}_{PC_CHARGE:+.1f}_{d}', xyz, q, folder, nroots=5, extra=f'%pointcharges "{pcfile}"\n')
    e, lam, f = lowest_bright(excitations(out))
    rows.append({'distance (Å)': d, 'E (eV)': e, 'wavelength (nm)': lam, 'f': f})
pcs = pd.DataFrame(rows)
display(pcs.round(3))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(PC_DISTANCES, pcs['wavelength (nm)'][1:], 'o-', label=f'charge {PC_CHARGE:+.0f} near atom {PC_ATOM}')
ax.axhline(reference[1], color='grey', ls='--', label='no charge')
ax.set_xlabel('distance of the charge from the atom (Å)'); ax.set_ylabel('wavelength of the first bright state (nm)'); ax.legend()
ax.set_title(PC_MOLECULE, loc='left')
plt.tight_layout(); save_figure(fig, f'g2_point_charge_{PC_MOLECULE}_{PC_ATOM}'); plt.show()

*Your text here.*

### 6.2 Molecules move: the shape of the absorption band

Our spectra were computed for one structure, the minimum of the energy, and the bands were drawn with an arbitrary width. In reality, at room temperature the molecule vibrates: bond lengths and twists change all the time, and each structure absorbs at a slightly different energy. To a large extent, the width of an absorption band comes from this motion.

Here we run a short molecular dynamics of the chromophore at 300 K with GFN2-xTB (as in Part 5 of the training), take snapshots along it, and compute the excitation energies of each snapshot. The spectrum is the average of the spectra of the snapshots (with 10 snapshots it is still noisy: more snapshots give a smoother band). **Time**: about 5 minutes for the dynamics, plus `N_SNAPSHOTS` TD-DFT calculations (1 minute each for the short chains, 5 for HBDI).

*How large is the spread of the excitation energies (in eV, and in nm)? Is the average different from the value at the minimum? Compare the width of the band obtained in this way with the width we assumed before (0.3 eV) and with the width of the experimental band of your molecule. Which motions change the excitation energy the most (compare the snapshots with the highest and the lowest energy)? What is missing in this approach? (Think of the quantum nature of the nuclei, and of the solvent.)*

In [ ]:
%%group 2
MD_MOLECULE = my_molecules[0]
N_SNAPSHOTS = 10

folder = os.path.join('g2', MD_MOLECULE, 'dynamics')
q = charge_of(MD_MOLECULE)
xyz = os.path.join('g2', MD_MOLECULE, MD_MOLECULE + '_opt', MD_MOLECULE + '_opt.xyz')
md = f'''! Native-GFN2-XTB MD
%md
  Timestep 0.5_fs
  Randomize 1
  Initvel 300_K
  Thermostat Berendsen 300_K Timecon 50_fs
  Dump Position Stride 100 Filename "trajectory.xyz"
  Run {200 * (N_SNAPSHOTS + 3)}
end
* xyz {q} 1
{xyz_block(xyz)}
*'''
run_orca(MD_MOLECULE + '_md', md, folder=folder)
lines = open(os.path.join(folder, MD_MOLECULE + '_md', 'trajectory.xyz')).read().splitlines()
n = int(lines[0])
frames = [lines[i:i + n + 2] for i in range(0, len(lines) - n - 1, n + 2)]
frames = frames[6::2][:N_SNAPSHOTS]          # skip the first 300 fs, then one snapshot every 100 fs

snapshots = []
for k, frame in enumerate(frames):
    snap = os.path.join(folder, f'snapshot_{k}.xyz')
    with open(snap, 'w') as f:
        f.write('\n'.join(frame) + '\n')
    snapshots.append(excitations(tddft(f'{MD_MOLECULE}_snapshot_{k}', snap, q, folder, nroots=5)))
bright = np.array([lowest_bright(s) for s in snapshots])
minimum = lowest_bright(json.load(open(os.path.join('g2', 'results', MD_MOLECULE + '.json')))['states']['CAM-B3LYP vacuum'])
print(f'first bright state: at the minimum {minimum[0]:.2f} eV; in the snapshots {bright[:, 0].mean():.2f} ± {bright[:, 0].std():.2f} eV '
      f'(from {bright[:, 0].min():.2f} to {bright[:, 0].max():.2f} eV)')

fig, ax = plt.subplots(figsize=(7, 4))
emin, emax = minimum[0] - 1.5, minimum[0] + 1.5
E = np.linspace(emin, emax, 1000)
average = sum(spectrum(s, fwhm=0.1, emin=emin, emax=emax)[1] for s in snapshots) / len(snapshots)   # narrow bands: the width comes from the snapshots
ax.plot(E, average, label=f'average of {len(snapshots)} snapshots at 300 K')
ax.plot(*spectrum(np.array([minimum]), fwhm=0.3, emin=emin, emax=emax), '--', label='minimum, width 0.3 eV')
ax.vlines(bright[:, 0], 0, 0.3 * average.max(), color='grey', lw=1)
ax.set_xlabel('excitation energy (eV)'); ax.set_ylabel('absorption (arb. units)'); ax.legend(); ax.set_title(MD_MOLECULE, loc='left')
plt.tight_layout(); save_figure(fig, f'g2_dynamics_{MD_MOLECULE}'); plt.show()

*Your text here.*